# Track A — Active SINDy Dynamics + Iterative RL Controller

SINDy dynamics and a control policy **co-improve** in an iterative active learning loop:

```
Bootstrap data  ──►  Fit SINDy  ──►  Train policy in surrogate (imagination)
                                              │
                                    Execute policy in real MuJoCo sim
                                    + collect near-equilibrium data
                                              │
                                    Refit SINDy on expanded dataset
                                              └──► repeat until convergence
```

**Key motivation:** random-policy episodes terminate in ~10 steps and never reach
the near-equilibrium region that SINDy must model accurately. A trained policy
generates exactly the trajectories SINDy benefits from most.

## Algorithm Choice: DreamerV3-style Imagination

The iterative loop maps exactly onto the DreamerV3 paradigm:

| DreamerV3 concept | This notebook |
|---|---|
| RSSM (recurrent world model) | SINDy surrogate (explicit sparse model) |
| Policy training via imagined rollouts | PPO inside `SINDySurrogateEnv` |
| World model update from real env | SINDy refit on expanded dataset |
| Real-env interaction | Controller rollout in MuJoCo |

SINDy replaces the RSSM with an **interpretable** sparse dynamics model.
At each iteration the policy "dreams" entirely inside the surrogate — no additional
real-simulator calls during PPO training.

In [ ]:
# uv add stable-baselines3   (if not already in pyproject.toml)
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pysindy as ps
import pathlib

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

# Derive timestep from environment once
_env = gym.make("InvertedDoublePendulum-v5")
DT   = _env.unwrapped.dt
_env.close()

STATE_DIM  = 6
ACTION_DIM = 1
SINDY_DEG  = 2
SINDY_THRS = 0.05

N_BOOTSTRAP = 50     # episodes for bootstrap (iteration 0)
N_ACTIVE    = 100    # episodes to collect with trained controller per iteration
PPO_STEPS   = 200_000
MAX_ITER    = 2      # active learning iterations

DATA_DIR  = pathlib.Path("../data")
MODEL_DIR = pathlib.Path("../results/trackA")
DATA_DIR.mkdir(exist_ok=True, parents=True)
MODEL_DIR.mkdir(exist_ok=True, parents=True)

BOOTSTRAP_PATH = DATA_DIR / "trajectories_trackA_iter0.npz"
STATE_LABELS   = ["x", "θ₁", "θ₂", "ẋ", "θ̇₁", "θ̇₂"]

print(f"DT = {DT:.4f} s  ({1/DT:.0f} Hz)")

In [ ]:
def obs_to_state6(obs):
    """9-dim MuJoCo observation → 6-dim state [x, θ₁, θ₂, ẋ, θ̇₁, θ̇₂].

    Angles recovered from sin/cos pairs via arctan2 (handles all quadrants).
    Velocities are taken directly from obs[5:8] (already clipped ±10 by MuJoCo).
    obs[8] (constraint force) is dropped — not part of the dynamics state.
    """
    return np.array([
        obs[0],
        np.arctan2(obs[1], obs[3]),   # θ₁ = arctan2(sin θ₁, cos θ₁)
        np.arctan2(obs[2], obs[4]),   # θ₂ = arctan2(sin θ₂, cos θ₂)
        obs[5],
        obs[6],
        obs[7],
    ])

## Iteration 0: Bootstrap Data Collection

In [ ]:
def balance_policy(obs, noise_std=0.1):
    """Simple PD balance heuristic for near-equilibrium bootstrap data.

    Applies a corrective force opposing the lean of both poles.
    Added Gaussian noise diversifies the state coverage.
    """
    u = (-0.5 * obs[0]       # cart position
         - 3.0 * obs[1]      # sin(θ₁) ≈ θ₁ near upright
         - 1.0 * obs[2]      # sin(θ₂) ≈ θ₂ near upright
         - 0.5 * obs[5]      # ẋ damping
         - 0.5 * obs[6]      # θ̇₁ damping
         - 0.2 * obs[7])     # θ̇₂ damping
    u += np.random.normal(0, noise_std)
    return np.array([np.clip(u, -1.0, 1.0)], dtype=np.float32)


def random_policy(obs):
    return np.random.uniform(-1.0, 1.0, size=(1,)).astype(np.float32)


def collect_episodes(n_episodes, policy_fn, seed=0, label=""):
    """Collect (X, U, X_next) transitions using policy_fn(obs) -> action."""
    env = gym.make("InvertedDoublePendulum-v5")
    rng = np.random.default_rng(seed)
    X, U, X_next, lengths = [], [], [], []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=int(rng.integers(0, 2**31)))
        s      = obs_to_state6(obs)
        done   = False
        ep_len = 0

        while not done:
            a    = policy_fn(obs)
            nobs, _, term, trunc, _ = env.step(a)
            done = term or trunc
            ns   = obs_to_state6(nobs)
            X.append(s);  U.append(a.astype(np.float64));  X_next.append(ns)
            s = ns;  obs = nobs;  ep_len += 1

        lengths.append(ep_len)

    env.close()
    if label:
        print(f"  [{label}] {n_episodes} eps | {len(X):,} transitions"
              f" | median length {np.median(lengths):.0f} steps")
    return np.array(X), np.array(U), np.array(X_next)

In [ ]:
if BOOTSTRAP_PATH.exists():
    d = np.load(BOOTSTRAP_PATH)
    X_all, U_all, Xn_all = d["X"], d["U"], d["X_next"]
    print(f"Loaded bootstrap data ({X_all.shape[0]:,} transitions)")
else:
    print("Collecting bootstrap data (balance policy + random perturbations) …")
    X_all, U_all, Xn_all = collect_episodes(
        N_BOOTSTRAP, balance_policy, seed=0, label="iter-0 balance"
    )
    # Augment with random-policy episodes for off-equilibrium coverage
    X_r, U_r, Xn_r = collect_episodes(
        20, random_policy, seed=1, label="iter-0 random"
    )
    X_all  = np.vstack([X_all,  X_r])
    U_all  = np.vstack([U_all,  U_r])
    Xn_all = np.vstack([Xn_all, Xn_r])
    np.savez(BOOTSTRAP_PATH, X=X_all, U=U_all, X_next=Xn_all)
    print(f"Bootstrap dataset: {X_all.shape[0]:,} transitions saved.")

## Fit Initial SINDy Model (Iteration 0)

In [ ]:
def fit_sindy(X, U, X_next, degree=SINDY_DEG, threshold=SINDY_THRS):
    """Fit SINDyC surrogate. Returns fitted model and one-step RMSE on held-out 20%."""
    dX    = (X_next - X) / DT
    model = ps.SINDy(
        feature_library=ps.PolynomialLibrary(degree=degree),
        optimizer=ps.optimizers.STLSQ(threshold=threshold, alpha=0.01),
    )
    model.fit(x=X, t=DT, u=U, x_dot=dX)

    n  = len(X)
    te = np.random.default_rng(7).choice(n, n // 5, replace=False)
    Xn_pred = X[te] + DT * model.predict(X[te], u=U[te])
    rmse    = float(np.sqrt(np.mean((Xn_pred - X_next[te]) ** 2)))
    return model, rmse


sindy_models = {}
rmse_history = {}

sindy_models[0], rmse_history[0] = fit_sindy(X_all, U_all, Xn_all)
print(f"Iteration 0  |  transitions: {X_all.shape[0]:,}  |  one-step RMSE: {rmse_history[0]:.6f}")

## SINDy Surrogate Environment

> **Coordination note (Track A & B):** Both tracks need `SINDySurrogateEnv`.
> The interface spec here matches Track B's wrapper exactly.
> Once both tracks are running, consolidate into `sindy_rl/envs/sindy_env.py`
> and import from there.

In [ ]:
L1, L2 = 0.6, 0.6  # pole lengths (m) — from MuJoCo inverted_double_pendulum.xml

from gymnasium import spaces

class SINDySurrogateEnv(gym.Env):
    """Gymnasium env backed by a fitted PySINDy model.

    Mirrors InvertedDoublePendulum-v5:
      obs  : Box(-inf, inf, (9,), float64)
      act  : Box(-1, 1, (1,), float32)
      done : tip height <= 1 m  |  |cart x| > 0.2 m  |  step >= max_steps

    Reward replicates MuJoCo formula (verified against Phase 1 intro notebook):
      dist  = 0.01 * x_tip^2 + (y_tip - 2)^2
      vel   = 1e-3 * clip(xdot)^2 + 5e-3 * clip(th1dot)^2 + 5e-3 * clip(th2dot)^2
      r     = 10.0 - dist - vel
    """
    metadata = {"render_modes": []}

    def __init__(self, sindy_model, initial_states, dt=DT, max_steps=1000):
        super().__init__()
        self.sindy_model    = sindy_model
        self.initial_states = np.asarray(initial_states)
        self.dt             = dt
        self.max_steps      = max_steps
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(9,), dtype=np.float64)
        self.action_space      = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)
        self._s = None
        self._n = 0

    @staticmethod
    def _state_to_obs(s):
        x, th1, th2, xdot, th1dot, th2dot = s
        return np.array([
            x,
            np.sin(th1), np.sin(th2),
            np.cos(th1), np.cos(th2),
            np.clip(xdot,   -10.0, 10.0),
            np.clip(th1dot, -10.0, 10.0),
            np.clip(th2dot, -10.0, 10.0),
            0.0,   # constraint force — zero in surrogate
        ], dtype=np.float64)

    def _reward(self):
        x, th1, th2, xdot, th1dot, th2dot = self._s
        y_tip = L1 * np.cos(th1) + L2 * np.cos(th1 + th2)
        x_tip = x  + L1 * np.sin(th1) + L2 * np.sin(th1 + th2)
        dist  = 0.01 * x_tip**2 + (y_tip - 2.0)**2
        vel   = (1e-3 * np.clip(xdot,   -10.0, 10.0)**2
               + 5e-3 * np.clip(th1dot, -10.0, 10.0)**2
               + 5e-3 * np.clip(th2dot, -10.0, 10.0)**2)
        return float(10.0 - dist - vel)

    def _terminated(self):
        if not np.all(np.isfinite(self._s)):
            return True
        x, th1, th2 = self._s[0], self._s[1], self._s[2]
        if L1 * np.cos(th1) + L2 * np.cos(th1 + th2) <= 1.0:
            return True
        if abs(x) > 0.2:
            return True
        return False

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        idx      = self.np_random.integers(0, len(self.initial_states))
        self._s  = self.initial_states[idx].copy()
        self._s += self.np_random.normal(0.0, 0.01, STATE_DIM)
        self._n  = 0
        return self._state_to_obs(self._s), {}

    def step(self, action):
        a    = np.clip(np.asarray(action, dtype=np.float64).reshape(1), -1.0, 1.0)
        xdot = self.sindy_model.predict(self._s.reshape(1, -1), u=a.reshape(1, -1))[0]
        self._s  = self._s + self.dt * xdot
        r        = self._reward()
        done     = self._terminated()
        self._n += 1
        return self._state_to_obs(self._s), r, done, self._n >= self.max_steps, {}

    def render(self): pass


In [ ]:
from gymnasium.utils.env_checker import check_env

_test_env = SINDySurrogateEnv(sindy_models[0], X_all[:200])
check_env(_test_env, warn=True)
print("env_checker: OK")

# Quick smoke test
obs, _ = _test_env.reset(seed=0)
for _ in range(10):
    obs, r, term, trunc, _ = _test_env.step(_test_env.action_space.sample())
print(f"10 random steps completed — final reward: {r:.4f}")

## Iterative Active Learning Loop
### DreamerV3 mapping: train policy in imagination → execute in real env → update world model

Each iteration:
1. **Train PPO in SINDy surrogate** (= "dreaming" — no real-sim calls)
2. **Execute trained policy in real MuJoCo sim** (= real-env interaction)
3. **Refit SINDy on expanded dataset** (= world model update)

In [ ]:
def rollout_sindy(model, x0, U_seq):
    """Roll out SINDy model from x0 for len(U_seq) steps (Euler integration)."""
    x    = x0.copy()
    traj = [x]
    for u in U_seq:
        xdot = model.predict(x.reshape(1, -1), u=u.reshape(1, -1))[0]
        x    = x + DT * xdot
        traj.append(x)
    return np.array(traj)


def train_ppo_in_surrogate(sindy_model, dataset, n_steps, iteration):
    """Train PPO inside SINDy surrogate. Returns the trained policy."""
    env   = Monitor(SINDySurrogateEnv(sindy_model, dataset))
    model = PPO(
        "MlpPolicy", env,
        learning_rate=3e-4, n_steps=2048,
        batch_size=64, n_epochs=10,
        gamma=0.99, ent_coef=0.01,
        verbose=0,
    )
    model.learn(total_timesteps=n_steps, progress_bar=True)
    path = str(MODEL_DIR / f"ppo_iter{iteration}")
    model.save(path)
    recent = env.get_episode_rewards()[-100:]
    print(f"  Surrogate final-100-ep mean reward: {np.mean(recent):.2f}")
    return model


def collect_with_policy(ppo_model, n_episodes, seed=0, label=""):
    """Execute trained policy in real MuJoCo sim. DreamerV3 real-env phase."""
    def policy_fn(obs):
        action, _ = ppo_model.predict(obs, deterministic=True)
        return action.astype(np.float32)
    return collect_episodes(n_episodes, policy_fn, seed=seed, label=label)

In [ ]:
print("=" * 60)
print("Active SINDy learning loop")
print("=" * 60)

for iteration in range(1, MAX_ITER + 1):
    print(f"\n── Iteration {iteration} " + "─" * 44)

    # Step 1: Train policy inside SINDy surrogate (dreaming phase)
    print(f"  Step 1: Train PPO in SINDy[{iteration-1}] surrogate ({PPO_STEPS:,} steps)")
    ppo = train_ppo_in_surrogate(
        sindy_models[iteration - 1], X_all, PPO_STEPS, iteration
    )

    # Step 2: Collect near-equilibrium data with trained policy in real sim
    print(f"  Step 2: Execute policy in real sim — collect {N_ACTIVE} episodes")
    X_new, U_new, Xn_new = collect_with_policy(
        ppo, N_ACTIVE,
        seed=iteration * 100,
        label=f"iter-{iteration} policy rollout",
    )

    # Step 3: Expand dataset and refit SINDy (world model update)
    X_all  = np.vstack([X_all,  X_new])
    U_all  = np.vstack([U_all,  U_new])
    Xn_all = np.vstack([Xn_all, Xn_new])
    np.savez(
        DATA_DIR / f"trajectories_trackA_iter{iteration}.npz",
        X=X_all, U=U_all, X_next=Xn_all,
    )

    print(f"  Step 3: Refit SINDy on {X_all.shape[0]:,} transitions")
    sindy_models[iteration], rmse_history[iteration] = fit_sindy(X_all, U_all, Xn_all)
    print(
        f"  One-step RMSE: {rmse_history[iteration]:.6f}"
        f"  (was {rmse_history[iteration-1]:.6f},"
        f"  Δ = {rmse_history[iteration] - rmse_history[iteration-1]:+.6f})"
    )

print("\nLoop complete.")

## Results: SINDy Model Improvement Over Iterations

In [ ]:
iters     = sorted(rmse_history.keys())
rmse_vals = [rmse_history[i] for i in iters]

plt.figure(figsize=(7, 4))
plt.plot(iters, rmse_vals, "o-", lw=2, markersize=8)
plt.xticks(iters, [f"Iter {i}" for i in iters])
plt.ylabel("One-step RMSE")
plt.title("Track A — SINDy one-step RMSE vs active learning iteration")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

for i in iters:
    n = np.load(DATA_DIR / f"trajectories_trackA_iter{i}.npz")["X"].shape[0] if i > 0 else X_all.shape[0]
    print(f"  Iter {i}: RMSE = {rmse_history[i]:.6f}  |  cumulative transitions ≈ {n:,}")

## Final Evaluation in Real Simulator

In [ ]:
final_iter = max(sindy_models.keys())
final_ppo  = PPO.load(str(MODEL_DIR / f"ppo_iter{final_iter}"))

eval_env = gym.make("InvertedDoublePendulum-v5")
N_EVAL   = 20
ep_rewards_real, ep_lengths_real = [], []

for seed in range(N_EVAL):
    obs, _ = eval_env.reset(seed=seed)
    done = False
    ep_r = ep_l = 0
    while not done:
        action, _ = final_ppo.predict(obs, deterministic=True)
        obs, r, term, trunc, _ = eval_env.step(action)
        done  = term or trunc
        ep_r += r
        ep_l += 1
    ep_rewards_real.append(ep_r)
    ep_lengths_real.append(ep_l)

eval_env.close()

mean_r  = np.mean(ep_rewards_real)
std_r   = np.std(ep_rewards_real)
mean_l  = np.mean(ep_lengths_real)
surv500 = 100 * np.mean(np.array(ep_lengths_real) > 500)
n_boot  = np.load(BOOTSTRAP_PATH)["X"].shape[0]
n_act   = X_all.shape[0] - n_boot

print("── Track A: Final evaluation (real simulator) ─────────────────────")
print(f"  Policy from iteration        : {final_iter}")
print(f"  Mean episode reward          : {mean_r:.2f} ± {std_r:.2f}")
print(f"  Mean episode duration        : {mean_l:.0f} steps")
print(f"  % surviving > 500 steps      : {surv500:.0f} %")
print(f"  Bootstrap sim interactions   : {n_boot:,}")
print(f"  Active-loop sim interactions : {n_act:,}")
print(f"  Total real sim interactions  : {X_all.shape[0]:,}")
print(f"  SINDy RMSE (final)           : {rmse_history[final_iter]:.6f}")